# WSA_04 — FSA & Horizon Comparison

**Purpose.** Compare scenario sensitivity across all six FSAs and all 24 forecast horizons.

> Run from the project repository. Outputs are generated only from the project data and frozen model artifacts.

In [1]:
# Import libraries
from pathlib import Path
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt


In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG = PROJECT_ROOT / 'configs' / 'weather_sensitivity.yaml'
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/weather_sensitivity.yaml')

In [3]:
# Import modules for weather_sensitivity
from src.ontario_peak_risk.weather_sensitivity.common import load_config, ensure_dirs

In [4]:
cfg, project_root = load_config(CONFIG)
paths = ensure_dirs(cfg, project_root)

print('Project root:', project_root)


Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk


In [5]:
from src.ontario_peak_risk.weather_sensitivity.comparison import (
    summary_by_fsa_scenario,
    symmetry_table,
)

master_path = paths['outputs_dir'] / 'WSA_sensitivity_master.parquet'
if not master_path.exists():
    raise FileNotFoundError(
        'WSA_sensitivity_master.parquet not found. Run WSA_02 successfully first.'
    )

results = pd.read_parquet(master_path)

required_cols = [
    'fsa', 'horizon', 'scenario', 'temperature_delta_c',
    'forecast_delta_kwh', 'forecast_delta_pct',
    'peak_risk_delta', 'alert_changed'
]
missing_cols = [c for c in required_cols if c not in results.columns]
if missing_cols:
    raise KeyError(f'Missing expected WSA_04 columns: {missing_cols}')

expected_rows = (
    len(cfg['analysis']['fsas'])
    * int(cfg['analysis']['horizons'])
    * len(cfg['analysis']['scenarios_c'])
)
if len(results) != expected_rows:
    raise ValueError(
        f'Unexpected master row count: {len(results)}; expected {expected_rows}.'
    )

# Baseline rows should remain unchanged by definition.
baseline = results.loc[results['temperature_delta_c'].eq(0)].copy()
if baseline.empty:
    raise ValueError('Baseline scenario was not found in the master sensitivity table.')

if not np.allclose(baseline['forecast_delta_kwh'].fillna(0), 0.0):
    raise ValueError('Baseline forecast deltas are not all zero.')
if not np.allclose(baseline['peak_risk_delta'].fillna(0), 0.0):
    raise ValueError('Baseline Peak-Risk deltas are not all zero.')

summary = summary_by_fsa_scenario(results)
symmetry = symmetry_table(results)

summary_path = paths['outputs_dir'] / 'WSA_04_fsa_scenario_summary.csv'
symmetry_path = paths['outputs_dir'] / 'WSA_04_symmetry_analysis.csv'

summary.to_csv(summary_path, index=False)
symmetry.to_csv(symmetry_path, index=False)

display(summary)



,fsa,scenario,temperature_delta_c,mean_abs_forecast_delta_kwh,max_abs_forecast_delta_kwh,mean_abs_forecast_delta_pct,mean_abs_peak_risk_delta,max_abs_peak_risk_delta,alert_changes,ood_rows,caution_rows
0,L4T,+2.5C,2.5,77.294838,205.984713,0.796846,0.003478,0.032620,0,0,0
1,L4T,+5C,5.0,124.085796,293.257204,1.253187,0.017699,0.150898,3,0,0
2,L4T,-2.5C,-2.5,117.164462,267.846435,1.206130,0.004959,0.054660,0,0,0
3,L4T,-5C,-5.0,291.712413,633.727903,2.983863,0.004098,0.045094,0,0,0
4,L4T,Baseline,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0
5,M5R,+2.5C,2.5,28.673148,104.755046,0.289926,0.077564,0.386207,2,0,0
6,M5R,+5C,5.0,61.628220,158.888097,0.619037,0.120955,0.495182,3,0,0
7,M5R,-2.5C,-2.5,84.391206,222.694394,0.859796,0.048007,0.207717,3,0,0
8,M5R,-5C,-5.0,285.652038,698.611438,2.894259,0.078999,0.313939,4,0,0
9,M5R,Baseline,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0


In [6]:
symmetry_summary = (
    symmetry.groupby(['metric', 'magnitude_c'], observed=True)['symmetry_residual']
    .agg(['mean', 'median', 'min', 'max'])
    .reset_index()
)
display(symmetry_summary)

print('Rows:', len(results))
print('Expected rows:', expected_rows)
print('Baseline validation: PASS')
print('FSA/scenario comparison: PASS')


,metric,magnitude_c,mean,median,min,max
0,forecast_delta_kwh,2.5,42.935190,22.450819,-113.061734,519.889250
1,forecast_delta_kwh,5.0,150.890255,89.001663,-69.498724,709.248410
2,peak_risk_delta,2.5,-0.015189,0.000068,-0.335716,0.219820
3,peak_risk_delta,5.0,-0.049855,0.000040,-0.481930,0.263272


Rows: 720
Expected rows: 720
Baseline validation: PASS
FSA/scenario comparison: PASS


In [7]:
horizon = (
    results.groupby(
        ['horizon', 'scenario', 'temperature_delta_c'],
        observed=True,
    )
    .agg(
        mean_abs_forecast_delta_kwh=(
            'forecast_delta_kwh', lambda s: s.abs().mean()
        ),
        mean_abs_peak_risk_delta=(
            'peak_risk_delta', lambda s: s.abs().mean()
        ),
        alert_changes=('alert_changed', 'sum'),
    )
    .reset_index()
    .sort_values(['horizon', 'temperature_delta_c'])
)

expected_horizon_rows = (
    int(cfg['analysis']['horizons'])
    * len(cfg['analysis']['scenarios_c'])
)
if len(horizon) != expected_horizon_rows:
    raise ValueError(
        f'Unexpected horizon-summary row count: {len(horizon)}; '
        f'expected {expected_horizon_rows}.'
    )

horizon_path = paths['outputs_dir'] / 'WSA_04_horizon_summary.csv'
horizon.to_csv(horizon_path, index=False)

display(horizon.head(15))
print('Horizon-summary rows:', len(horizon))
print('Expected horizon-summary rows:', expected_horizon_rows)
print('WSA_04 RESULT: COMPLETE')


,horizon,scenario,temperature_delta_c,mean_abs_forecast_delta_kwh,mean_abs_peak_risk_delta,alert_changes
3,1,-5C,-5.0,426.236454,0.001013,0
2,1,-2.5C,-2.5,158.015064,0.000330,0
4,1,Baseline,0.0,0.000000,0.000000,0
0,1,+2.5C,2.5,116.318690,0.000061,0
1,1,+5C,5.0,163.049672,0.000076,0
8,2,-5C,-5.0,451.138607,0.000198,0
7,2,-2.5C,-2.5,180.105898,0.000191,0
9,2,Baseline,0.0,0.000000,0.000000,0
5,2,+2.5C,2.5,98.478868,0.000052,0
6,2,+5C,5.0,173.197685,0.000095,0


Horizon-summary rows: 120
Expected horizon-summary rows: 120
WSA_04 RESULT: COMPLETE
